# Default Classification Decision Tree

Baseline notebook for training a default `DecisionTreeClassifier` on `Tommy_Award_Player_Game_Table_hustle.csv` using `y` as the target.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

In [ ]:
csv_path = "Tommy_Award_Player_Game_Table_hustle.csv"
target_col = "y"
selected_test_seasons = ["2024-25", "2025-26"]
random_state = 42

In [ ]:
df = pd.read_csv(csv_path)

if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found in {csv_path}.")

if "GAME_ID" in df.columns:
    game_id_col = "GAME_ID"
elif "gameId" in df.columns:
    game_id_col = "gameId"
else:
    raise ValueError("Could not find a game id column (expected GAME_ID or gameId).")

if "season" not in df.columns:
    if "game_date" not in df.columns:
        raise ValueError("Need either 'season' or 'game_date' column for season-based split.")

    game_dates = pd.to_datetime(df["game_date"], errors="coerce")

    def to_season_label(ts: pd.Timestamp) -> str:
        if pd.isna(ts):
            return "unknown"
        start_year = ts.year if ts.month >= 10 else ts.year - 1
        return f"{start_year}-{str((start_year + 1) % 100).zfill(2)}"

    df["season"] = game_dates.map(to_season_label)

df.head()

In [ ]:
# Drop obvious label-leakage columns when present.
leakage_columns = {"winner_names"}
existing_leakage_columns = [c for c in leakage_columns if c in df.columns]
if existing_leakage_columns:
    df = df.drop(columns=existing_leakage_columns)

test_mask = df["season"].isin(selected_test_seasons)
train_df = df.loc[~test_mask].copy()
test_df = df.loc[test_mask].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Train or test split is empty. Check selected_test_seasons and season values.")

feature_cols = [c for c in df.columns if c != target_col]
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df[target_col].astype(int)
y_test = test_df[target_col].astype(int)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Rows total: {len(df):,}")
print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Test seasons: {', '.join(selected_test_seasons)}")
print(f"Features used: {len(feature_cols)}")

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=random_state)),
    ]
)

In [ ]:
model.fit(X_train, y_train)

scored_df = test_df.copy()
scored_df["pred_prob"] = model.predict_proba(X_test)[:, 1]

winner_idx = scored_df.groupby(game_id_col)["pred_prob"].idxmax()
y_pred_top1 = pd.Series(0, index=scored_df.index, dtype=int)
y_pred_top1.loc[winner_idx] = 1

In [ ]:
metrics = {
    "game_top1_accuracy": scored_df.loc[winner_idx, target_col].mean(),
    "row_f1": f1_score(y_test, y_pred_top1, zero_division=0),
    "row_recall": recall_score(y_test, y_pred_top1, zero_division=0),
    "row_prauc": average_precision_score(y_test, scored_df["pred_prob"]),
    "row_precision": precision_score(y_test, y_pred_top1, zero_division=0),
}

pd.DataFrame([metrics])